# Osnovni eksperiment: eksplozija susedstva i uzorkovanje

Ponovljeno merenje osnovnog eksperimenta na nacionalnoj platformi, na cvoru sa karticom NVIDIA A100.

Notebook sam prolazi kroz sva tri seeda. Rezultat svake konfiguracije se upisuje odmah, pa se pri ponovnom pokretanju preskace ono sto je vec izmereno.

Ispravke u odnosu na prvo merenje:

1. Graf ogbn-arxiv se pretvara u neusmeren, kako radi i zvanicni OGB primer. Bez toga uzorkovanje prati samo ulazne potege pa model za veliki deo cvorova ne vidi nijednog suseda.
2. Vrh memorije se cita posle trening petlje, a pre ocene nad celim grafom. U prvom merenju je citan posle ocene, pa je izmerena potrosnja ocene, ista za sve konfiguracije.
3. Statistika stepena i sirenje okoline racunaju se po ulaznim potezima, jer uzorkovanje u biblioteci ide u tom smeru.
4. Seed se postavlja iznova pre svake konfiguracije, pa su poredjenja uparena.
5. Za full-batch pristup graf stoji na kartici, a za uzorkovanje u radnoj memoriji. Izmerena memorija kartice zato za full-batch ukljucuje i sam graf, sto i jeste razlika izmedju dva pristupa.
6. Uz teorijski model d^L racuna se i granica koja uzima u obzir raspodelu stepena, jer prosecan stepen ne opisuje graf sa raspodelom po zakonu stepena.

Full-batch trening ide 500 epoha, kako radi zvanicni OGB primer, a uzorkovanje 50 epoha. Broj gradijentnih koraka se zato razlikuje i upisuje se u rezultate, da poredjenje ne bi bilo citano kao da su budzeti isti.

Procenjeno trajanje je oko sat vremena za sva tri seeda.

In [ ]:
SEEDOVI = [42, 123, 7]

In [ ]:
import os, sys, time, json, gc, csv, random, shutil, logging
from pathlib import Path
from datetime import datetime

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter, NullFormatter, LogLocator

from torch_geometric.nn import SAGEConv, GCNConv
from torch_geometric.loader import NeighborLoader
from torch_geometric.utils import to_undirected, degree, k_hop_subgraph
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator
import torch_geometric.data

torch.serialization.add_safe_globals([
    torch_geometric.data.data.DataEdgeAttr,
    torch_geometric.data.data.DataTensorAttr,
    torch_geometric.data.storage.GlobalStorage,
])

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda')

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
                    handlers=[logging.StreamHandler(sys.stdout)], force=True)
log = logging.getLogger(__name__)

cc = torch.cuda.get_device_capability(0)
log.info(f'kartica {torch.cuda.get_device_name(0)}, compute capability {cc[0]}.{cc[1]}, '
         f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
log.info(f'jezgara {os.cpu_count()}, deljena memorija {shutil.disk_usage("/dev/shm").total/1e9:.0f} GB')


def postavi_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def direktorijum(seed):
    p = Path(f'rezultati_osnovni/run_{seed}')
    p.mkdir(parents=True, exist_ok=True)
    return p


def upisi_red(seed, fajl, red):
    p = direktorijum(seed) / fajl
    novi = not p.exists()
    with open(p, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(red.keys()))
        if novi:
            w.writeheader()
        w.writerow(red)


def vec_uradjeno(seed, fajl, kolona):
    p = direktorijum(seed) / fajl
    if not p.exists():
        return set()
    with open(p) as f:
        return {r[kolona] for r in csv.DictReader(f)}


## Zajednicka podesavanja i modeli

In [ ]:
SKRIVENI, LR, DROPOUT = 256, 0.01, 0.5
FB_EPOHA, FB_OCENA_SVAKIH = 500, 25
UZ_EPOHA, UZ_BATCH, UZ_OCENA_SVAKIH = 50, 1024, 5
UCITAVANJE = dict(num_workers=8, persistent_workers=True, prefetch_factor=4, pin_memory=True)

E1_UZORAK, E1_DUBINE = 200, [1, 2, 3, 4]
E2_PO_GRUPI, E2_DUBINE = 50, [1, 2, 3]
E2_GRUPE = [('nizak', 1, 5), ('srednji', 6, 20), ('visok', 21, 100), ('vrlo visok', 101, 10**9)]

FULLBATCH = [('GCN', 2), ('GCN', 3), ('GCN', 4), ('GraphSAGE', 2), ('GraphSAGE', 3)]

UZORKOVANJE = [
    ('uzorak_5',      [5, 5]),
    ('uzorak_10',     [10, 10]),
    ('uzorak_25_10',  [25, 10]),
    ('svi_susedi',    [-1, -1]),
]


class GraphSAGE(torch.nn.Module):
    def __init__(self, ulaz, skriveni, izlaz, slojeva, dropout=DROPOUT):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        self.slojeva, self.dropout = slojeva, dropout
        self.convs.append(SAGEConv(ulaz, skriveni))
        self.bns.append(torch.nn.BatchNorm1d(skriveni))
        for _ in range(slojeva - 2):
            self.convs.append(SAGEConv(skriveni, skriveni))
            self.bns.append(torch.nn.BatchNorm1d(skriveni))
        self.convs.append(SAGEConv(skriveni, izlaz))

    def forward(self, x, edge_index):
        for i in range(self.slojeva - 1):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.convs[-1](x, edge_index)


class GCN(torch.nn.Module):
    def __init__(self, ulaz, skriveni, izlaz, slojeva, dropout=DROPOUT):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        self.slojeva, self.dropout = slojeva, dropout
        self.convs.append(GCNConv(ulaz, skriveni, cached=True))
        self.bns.append(torch.nn.BatchNorm1d(skriveni))
        for _ in range(slojeva - 2):
            self.convs.append(GCNConv(skriveni, skriveni, cached=True))
            self.bns.append(torch.nn.BatchNorm1d(skriveni))
        self.convs.append(GCNConv(skriveni, izlaz, cached=True))

    def forward(self, x, edge_index):
        for i in range(self.slojeva - 1):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.convs[-1](x, edge_index)


def napravi(ime, slojeva, ulaz, izlaz):
    razred = GCN if ime == 'GCN' else GraphSAGE
    return razred(ulaz, SKRIVENI, izlaz, slojeva).to(device)

## Ucitavanje grafa

Citatna mreza ogbn-arxiv je usmerena pa se pretvara u neusmerenu. Odnos broja potega pre i posle pretvaranja pokazuje koliko je veza bilo zapisano samo u jednom smeru.

In [ ]:
dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='podaci')
data = dataset[0]
data.y = data.y.squeeze()
split_idx = dataset.get_idx_split()
evaluator = Evaluator(name='ogbn-arxiv')

potega_usmereno = data.num_edges
data.edge_index = to_undirected(data.edge_index)
data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[split_idx['train']] = True

stepen = degree(data.edge_index[1], num_nodes=data.num_nodes)
d_prosek = stepen.mean().item()
d_bias = (stepen.pow(2).sum() / stepen.sum()).item()

log.info(f'ogbn-arxiv: {data.num_nodes:,} cvorova, {data.num_edges:,} potega '
         f'(usmereno je bilo {potega_usmereno:,})')
log.info(f'ulazni stepen: prosek {d_prosek:.2f}, medijana {stepen.median():.0f}, '
         f'najveci {stepen.max():.0f}')
log.info(f'ocekivani stepen suseda (E[d^2]/E[d]): {d_bias:.2f}')


@torch.no_grad()
def oceni(model, x=None, ei=None):
    model.eval()
    privremeno = x is None
    if privremeno:
        x = data.x.to(device, non_blocking=True)
        ei = data.edge_index.to(device, non_blocking=True)
    out = model(x, ei).argmax(dim=-1, keepdim=True).cpu()
    if privremeno:
        del x, ei
        torch.cuda.empty_cache()
    return {s: evaluator.eval({'y_true': data.y[split_idx[s]].unsqueeze(1),
                               'y_pred': out[split_idx[s]]})['acc']
            for s in ['train', 'valid', 'test']}

## Eksperiment 1: rast okoline sa dubinom

Za uzorak trening cvorova racuna se velicina L-hop okoline i poredi sa dva teorijska modela. Prvi je uobicajeni d^L sa prosecnim stepenom. Drugi uzima u obzir da se pri sirenju ne stize u prosecan cvor, nego u cvor srazmerno njegovom stepenu, pa je ocekivani stepen suseda E[d^2]/E[d].

Sirenje ide po ulaznim potezima, istim smerom kojim se siri i uzorkovanje.

In [ ]:
for SEED in SEEDOVI:
    uradjeno = vec_uradjeno(SEED, 'eksp1_okolina.csv', 'dubina')
    if len(uradjeno) == len(E1_DUBINE):
        continue
    postavi_seed(SEED)
    trening = split_idx['train']
    uzorak = trening[torch.randperm(len(trening))[:E1_UZORAK]]
    log.info('=' * 70)
    log.info(f'seed {SEED} | Eksperiment 1 | {len(uzorak)} cvorova')

    for L in E1_DUBINE:
        if str(L) in uradjeno:
            continue
        t0 = time.time()
        velicine = []
        for cvor in uzorak:
            podskup, _, _, _ = k_hop_subgraph(int(cvor), L, data.edge_index,
                                              relabel_nodes=False,
                                              num_nodes=data.num_nodes,
                                              flow='source_to_target')
            velicine.append(len(podskup))
        v = np.array(velicine, dtype=float)
        teor_dL = d_prosek ** L
        teor_stepen = d_prosek * (d_bias ** (L - 1))

        upisi_red(SEED, 'eksp1_okolina.csv', {
            'dubina': L, 'cvorova_u_uzorku': len(v),
            'prosek': float(v.mean()), 'medijana': float(np.median(v)),
            'maksimum': float(v.max()), 'minimum': float(v.min()),
            'pct_grafa': float(v.mean() / data.num_nodes * 100),
            'pct_grafa_maks': float(v.max() / data.num_nodes * 100),
            'teor_dL': teor_dL, 'teor_stepen': teor_stepen,
            'odnos_dL': float(v.mean() / teor_dL),
            'odnos_stepen': float(v.mean() / teor_stepen),
            'sekundi': time.time() - t0})
        log.info(f'  L={L}: prosek {v.mean():9.1f}  medijana {np.median(v):7.0f}  '
                 f'maks {v.max():7.0f}  {v.mean()/data.num_nodes*100:6.3f}% grafa  '
                 f'd^L={teor_dL:9.1f} (odnos {v.mean()/teor_dL:.2f})  '
                 f'sa stepenom={teor_stepen:10.1f} (odnos {v.mean()/teor_stepen:.2f})')

log.info('Eksperiment 1 zavrsen za sve seedove')

## Eksperiment 2: uticaj stepena cvora

Cvorovi se grupisu po ulaznom stepenu i za svaku grupu se meri velicina okoline. Razlika izmedju grupa pokazuje koliko posla nosi jedan uzorak u istom mini-batch-u.

In [ ]:
for SEED in SEEDOVI:
    uradjeno = vec_uradjeno(SEED, 'eksp2_stepen.csv', 'kljuc')
    postavi_seed(SEED)
    trening = split_idx['train']
    st_tr = stepen[trening]
    log.info('=' * 70)
    log.info(f'seed {SEED} | Eksperiment 2')

    for ime, donja, gornja in E2_GRUPE:
        maska = (st_tr >= donja) & (st_tr <= gornja)
        kandidati = trening[maska]
        if len(kandidati) == 0:
            log.info(f'  grupa {ime}: nema cvorova')
            continue
        izbor = kandidati[torch.randperm(len(kandidati))[:E2_PO_GRUPI]]
        for L in E2_DUBINE:
            kljuc = f'{ime}_L{L}'
            if kljuc in uradjeno:
                continue
            velicine = [len(k_hop_subgraph(int(c), L, data.edge_index,
                                           relabel_nodes=False,
                                           num_nodes=data.num_nodes,
                                           flow='source_to_target')[0])
                        for c in izbor]
            v = np.array(velicine, dtype=float)
            upisi_red(SEED, 'eksp2_stepen.csv', {
                'kljuc': kljuc, 'grupa': ime, 'stepen_od': donja,
                'stepen_do': gornja if gornja < 10**9 else -1,
                'dubina': L, 'cvorova_u_uzorku': len(v),
                'cvorova_u_grupi': int(maska.sum()),
                'prosek': float(v.mean()), 'medijana': float(np.median(v)),
                'pct_grafa': float(v.mean() / data.num_nodes * 100)})
            log.info(f'  {ime:11s} L={L}: {len(v):3d} cvorova, prosek {v.mean():9.1f}, '
                     f'{v.mean()/data.num_nodes*100:6.3f}% grafa')

log.info('Eksperiment 2 zavrsen za sve seedove')

## Eksperiment 3: full-batch trening

Ceo graf stoji na kartici i svaka epoha je jedan gradijentni korak. Vrh memorije se cita posle trening petlje a pre ocene, pa u njega ulazi i sam graf, koji kod full-batch pristupa mora biti rezidentan.

Mere se GCN na tri dubine i GraphSAGE na dve. GraphSAGE je tu da bi poredjenje sa uzorkovanjem u Eksperimentu 4 islo nad istim modelom, pa da razlika ne bi mesala dva efekta.

In [ ]:
for SEED in SEEDOVI:
    uradjeno = vec_uradjeno(SEED, 'eksp3_fullbatch.csv', 'oznaka')
    for ime, slojeva in FULLBATCH:
        oznaka = f'{ime}_L{slojeva}'
        if oznaka in uradjeno:
            continue
        postavi_seed(SEED)
        log.info('=' * 70)
        log.info(f'seed {SEED} | Eksperiment 3 | {oznaka}, {FB_EPOHA} epoha')

        x_gpu = data.x.to(device)
        ei_gpu = data.edge_index.to(device)
        y_gpu = data.y.to(device)
        maska = data.train_mask.to(device)

        model = napravi(ime, slojeva, data.num_features, dataset.num_classes)
        opt = torch.optim.Adam(model.parameters(), lr=LR)

        naj_val, naj_test, naj_epoha, vrh, vremena = 0.0, float('nan'), 0, 0.0, []
        for ep in range(FB_EPOHA):
            model.train()
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
            t0 = time.time()
            opt.zero_grad()
            out = model(x_gpu, ei_gpu)
            gubitak = F.cross_entropy(out[maska], y_gpu[maska])
            gubitak.backward()
            opt.step()
            torch.cuda.synchronize()
            vremena.append(time.time() - t0)
            vrh = max(vrh, torch.cuda.max_memory_allocated() / 1e6)
            if (ep + 1) % FB_OCENA_SVAKIH == 0 or ep == 0:
                a = oceni(model, x_gpu, ei_gpu)
                if a['valid'] > naj_val:
                    naj_val, naj_test, naj_epoha = a['valid'], a['test'], ep + 1
                log.info(f'  epoha {ep+1:3d}: val={a["valid"]*100:5.2f}% '
                         f'test={a["test"]*100:5.2f}% {vremena[-1]*1000:6.1f}ms {vrh:6.0f}MB')

        upisi_red(SEED, 'eksp3_fullbatch.csv', {
            'pristup': 'full-batch', 'oznaka': oznaka, 'model': ime, 'slojeva': slojeva,
            'epoha': FB_EPOHA, 'koraka_po_epohi': 1, 'ukupno_koraka': FB_EPOHA,
            'naj_val': naj_val, 'naj_test': naj_test, 'naj_epoha': naj_epoha,
            'vreme_po_epohi': float(np.mean(vremena)),
            'vreme_po_koraku_ms': float(np.mean(vremena)) * 1000,
            'vreme_treninga_s': float(np.sum(vremena)),
            'memorija_mb': vrh})
        log.info(f'seed {SEED} | {oznaka}: test={naj_test*100:.2f}% (epoha {naj_epoha}), '
                 f'{np.mean(vremena)*1000:.1f}ms/epohi, {np.sum(vremena):.0f}s ukupno, {vrh:.0f}MB')

        del model, opt, out, x_gpu, ei_gpu, y_gpu, maska
        gc.collect()
        torch.cuda.empty_cache()

log.info('Eksperiment 3 zavrsen za sve seedove')

## Eksperiment 4: trening sa uzorkovanjem suseda

Graf ostaje u radnoj memoriji, a na karticu ide samo podgraf tekuceg batch-a. Izmerena memorija kartice zato ne ukljucuje ceo graf, sto i jeste razlika u odnosu na full-batch pristup.

Konfiguracija svi_susedi uzima sve susede umesto uzorka. Ona koristi isti model kao ostale, pa razlika u odnosu na njih pokazuje sta donosi samo uzorkovanje.

In [ ]:
for SEED in SEEDOVI:
    uradjeno = vec_uradjeno(SEED, 'eksp4_uzorkovanje.csv', 'oznaka')
    for oznaka, S in UZORKOVANJE:
        if oznaka in uradjeno:
            continue
        postavi_seed(SEED)
        log.info('=' * 70)
        log.info(f'seed {SEED} | Eksperiment 4 | {oznaka}: S={S}, B={UZ_BATCH}')

        loader = NeighborLoader(data, num_neighbors=S, input_nodes=split_idx['train'],
                                batch_size=UZ_BATCH, shuffle=True, **UCITAVANJE)
        model = GraphSAGE(data.num_features, SKRIVENI, dataset.num_classes, len(S)).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=LR)

        naj_val, naj_test, naj_epoha, vrh, vremena = 0.0, float('nan'), 0, 0.0, []
        cvorova, potega = [], []
        for ep in range(UZ_EPOHA):
            model.train()
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
            t0 = time.time()
            for batch in loader:
                if ep == 0:
                    cvorova.append(batch.num_nodes)
                    potega.append(batch.num_edges)
                batch = batch.to(device, non_blocking=True)
                opt.zero_grad()
                out = model(batch.x, batch.edge_index)
                m = batch.train_mask[:batch.batch_size]
                if m.sum() == 0:
                    continue
                gubitak = F.cross_entropy(out[:batch.batch_size][m],
                                          batch.y[:batch.batch_size][m])
                gubitak.backward()
                opt.step()
            torch.cuda.synchronize()
            vremena.append(time.time() - t0)
            vrh = max(vrh, torch.cuda.max_memory_allocated() / 1e6)
            if (ep + 1) % UZ_OCENA_SVAKIH == 0 or ep == 0:
                a = oceni(model)
                if a['valid'] > naj_val:
                    naj_val, naj_test, naj_epoha = a['valid'], a['test'], ep + 1
                log.info(f'  epoha {ep+1:3d}: val={a["valid"]*100:5.2f}% '
                         f'test={a["test"]*100:5.2f}% {vremena[-1]:6.2f}s {vrh:6.0f}MB')

        upisi_red(SEED, 'eksp4_uzorkovanje.csv', {
            'pristup': 'uzorkovanje', 'oznaka': oznaka, 'model': 'GraphSAGE',
            'slojeva': len(S), 'S': str(S), 'batch': UZ_BATCH,
            'epoha': UZ_EPOHA, 'koraka_po_epohi': len(loader),
            'ukupno_koraka': len(loader) * UZ_EPOHA,
            'cvorova_po_batchu': float(np.mean(cvorova)),
            'potega_po_batchu': float(np.mean(potega)),
            'naj_val': naj_val, 'naj_test': naj_test, 'naj_epoha': naj_epoha,
            'vreme_po_epohi': float(np.mean(vremena)),
            'vreme_po_koraku_ms': float(np.mean(vremena)) / len(loader) * 1000,
            'vreme_treninga_s': float(np.sum(vremena)),
            'memorija_mb': vrh})
        log.info(f'seed {SEED} | {oznaka}: test={naj_test*100:.2f}% (epoha {naj_epoha}), '
                 f'{np.mean(cvorova):.0f} cvorova/batch, {np.mean(vremena):.2f}s/epohi, '
                 f'{np.sum(vremena):.0f}s ukupno, {vrh:.0f}MB')

        del model, opt, loader
        gc.collect()
        torch.cuda.empty_cache()

log.info('Eksperiment 4 zavrsen za sve seedove')


## Grafikoni i sazetak po seedu

In [ ]:
for SEED in SEEDOVI:
    d = direktorijum(SEED)
    e1 = pd.read_csv(d / 'eksp1_okolina.csv')
    e2 = pd.read_csv(d / 'eksp2_stepen.csv')
    e3 = pd.read_csv(d / 'eksp3_fullbatch.csv')
    e4 = pd.read_csv(d / 'eksp4_uzorkovanje.csv')

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.6))
    axes[0].plot(e1['dubina'], e1['prosek'], 'o-', label='izmereno')
    axes[0].plot(e1['dubina'], e1['teor_dL'], 's--', label='d^L')
    axes[0].plot(e1['dubina'], e1['teor_stepen'], '^--', label='sa raspodelom stepena')
    axes[0].set_yscale('log')
    axes[0].set_xlabel('dubina L'); axes[0].set_ylabel('cvorova u okolini')
    axes[0].set_title(f'Rast okoline, seed {SEED}')
    axes[0].set_xticks(e1['dubina']); axes[0].legend(); axes[0].grid(alpha=0.3)

    sve = pd.concat([e3, e4], ignore_index=True)
    boje = {'full-batch': 'tab:red', 'uzorkovanje': 'tab:blue'}
    znaci = ['o', 's', '^', 'v', 'D', 'P', 'X', '*', '<', '>']
    for ax, kol, xlab, nas in [
            (axes[1], 'memorija_mb', 'Memorija kartice (MB)', 'Tacnost i memorija'),
            (axes[2], 'vreme_treninga_s', 'Vreme treninga (s)', 'Tacnost i vreme')]:
        for k, (_, r) in enumerate(sve.iterrows()):
            ax.scatter(r[kol], r['naj_test'] * 100, s=110, c=boje[r['pristup']],
                       marker=znaci[k % len(znaci)], zorder=3,
                       label=f"{r['oznaka']} ({r['pristup']})")
        ax.set_xscale('log')
        ax.xaxis.set_major_locator(LogLocator(base=10, subs=(1, 2, 5), numticks=15))
        ax.xaxis.set_major_formatter(ScalarFormatter())
        ax.xaxis.set_minor_formatter(NullFormatter())
        ax.set_xlabel(xlab); ax.set_ylabel('Test tacnost (%)')
        ax.set_title(f'{nas}, seed {SEED}'); ax.grid(alpha=0.3)

    rukohvati, natpisi = axes[1].get_legend_handles_labels()
    fig.legend(rukohvati, natpisi, loc='lower center', ncol=5, fontsize=8, frameon=False)
    plt.tight_layout(rect=[0, 0.1, 1, 1])
    plt.savefig(d / 'osnovni.png', dpi=150, bbox_inches='tight')
    plt.show()

    json.dump({'seed': SEED, 'datum': datetime.now().isoformat(),
               'platforma': 'nacionalna platforma za vestacku inteligenciju',
               'kartica': torch.cuda.get_device_name(0),
               'compute_capability': f'{cc[0]}.{cc[1]}',
               'jezgara': os.cpu_count(),
               'deljena_memorija_gb': round(shutil.disk_usage('/dev/shm').total / 1e9, 1),
               'graf_neusmeren': True,
               'potega_usmereno': int(potega_usmereno),
               'potega_neusmereno': int(data.num_edges),
               'prosecan_stepen': round(d_prosek, 4),
               'ocekivani_stepen_suseda': round(d_bias, 4),
               'fb_epoha': FB_EPOHA, 'uz_epoha': UZ_EPOHA, 'uz_batch': UZ_BATCH,
               'skriveni': SKRIVENI, 'lr': LR},
              open(d / 'sazetak.json', 'w'), indent=2)

    log.info(f'--- seed {SEED} ---')
    for _, r in e1.iterrows():
        log.info(f"  okolina L={int(r['dubina'])}: prosek {r['prosek']:9.1f}, "
                 f"{r['pct_grafa']:6.3f}% grafa, odnos prema d^L {r['odnos_dL']:.2f}, "
                 f"prema granici sa stepenom {r['odnos_stepen']:.2f}")
    for _, r in sve.iterrows():
        log.info(f"  {r['oznaka']:15s} {r['pristup']:12s} test={r['naj_test']*100:5.2f}%  "
                 f"{r['vreme_treninga_s']:6.0f}s  {r['memorija_mb']:7.0f}MB  "
                 f"{int(r['ukupno_koraka']):6d} koraka")

log.info('Sve zavrseno')
